<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>
<h4>Система управления клиентами (Customer Management System)</h4>

----

### Вариант задания №11

<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Customer в C#, который будет представлять информацию о
клиентах или покупателях. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.<br><br>
Требования к базовому классу Customer:<br>
Атрибуты: Идентификатор клиента (CustomerId), Имя (Name), Электронная
почта (Email).<br><br>
Методы:<br>
GetFullName(): метод для получения полного имени клиента.<br>
UpdateEmail(string newEmail): метод для обновления электронной
почты клиента.<br>
ViewProfile(): метод для просмотра профиля клиента.<br>
Требования к производным классам:
1. VIPКлиент (VipCustomer): Должен содержать дополнительные атрибуты,
такие как Баланс лояльности (LoyaltyPoints). Метод ViewProfile() должен быть
переопределен для отображения дополнительной информации о VIPклиенте.
2. ОбычныйКлиент (RegularCustomer): Должен содержать дополнительные
атрибуты, такие как Дата регистрации (RegistrationDate).
Метод UpdateEmail() должен быть переопределен для добавления
информации о дате последнего обновления электронной почты.
3. ГрупповойКлиент (GroupCustomer) (если требуется третий класс): Должен
содержать дополнительные атрибуты, такие как Название группы
(GroupName). Метод GetFullName() должен быть переопределен для
отображения названия группы вместо имени клиента.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
// ИНТЕРФЕЙСЫ

public interface IDiscountable
{
    decimal DiscountPercent { get; }
    decimal ApplyDiscount(decimal price);
}

public interface INotifiable
{
    void Notify(string message);
}

public interface IPersonalService
{
    string PersonalManager { get; set; }
    void RequestConcierge(string request);
}

public interface ICsvExportable
{
    string Export();
}

public interface IJsonExportable
{
    string Export();
}

public interface ILogger
{
    void Log(string message);
}

public interface INotificationSender
{
    void Send(string to, string message);
}

// ДЕЛЕГАТЫ И КЛАССЫ АРГУМЕНТОВ СОБЫТИЙ

// Пользовательский делегат для фильтрации клиентов
public delegate bool CustomerFilter<T>(T customer)
    where T : Customer;

// Аргументы события изменения Email
public class EmailChangedEventArgs : EventArgs
{
    public string OldEmail { get; }
    public string NewEmail { get; }

    public EmailChangedEventArgs(string oldEmail, string newEmail)
    {
        OldEmail = oldEmail;
        NewEmail = newEmail;
    }
}

// Аргументы события изменения бонусных баллов
public class PointsChangedEventArgs : EventArgs
{
    public int OldPoints { get; }
    public int NewPoints { get; }
    public string Reason { get; }

    public PointsChangedEventArgs(
        int oldPoints, int newPoints, string reason)
    {
        OldPoints = oldPoints;
        NewPoints = newPoints;
        Reason = reason;
    }
}

// Аргументы событий коллекции клиентов
public class CustomerEventArgs<T> : EventArgs where T : Customer
{
    public T Customer { get; }

    public CustomerEventArgs(T customer)
    {
        Customer = customer;
    }
}

// КЛАССЫ ЛОГИРОВАНИЯ И УВЕДОМЛЕНИЙ

public class ConsoleLogger : ILogger
{
    public void Log(string message)
    {
        Console.WriteLine(
            $"[LOG {DateTime.Now:HH:mm:ss}] {message}");
    }
}

public class EmailSender : INotificationSender
{
    public void Send(string to, string message)
    {
        Console.WriteLine($"[EMAIL -> {to}] {message}");
    }
}

public class SmsSender : INotificationSender
{
    public void Send(string to, string message)
    {
        Console.WriteLine($"[SMS -> {to}] {message}");
    }
}

// ПРОСТОЙ DI-КОНТЕЙНЕР
// Используется коллекция Dictionary

public class ServiceContainer
{
    private readonly Dictionary<Type, Func<ServiceContainer, object>>
        _factories = new Dictionary<Type, Func<ServiceContainer, object>>();

    private readonly Dictionary<Type, object>
        _singletons = new Dictionary<Type, object>();

    public void RegisterSingleton<TService>(
        Func<ServiceContainer, TService> factory)
        where TService : class
    {
        _singletons.Remove(typeof(TService));

        _factories[typeof(TService)] = c =>
        {
            if (!_singletons.TryGetValue(
                typeof(TService), out var instance))
            {
                instance = factory(c);
                _singletons[typeof(TService)] = instance;
            }

            return instance;
        };
    }

    public void RegisterTransient<TService>(
        Func<ServiceContainer, TService> factory)
        where TService : class
    {
        _factories[typeof(TService)] = c => factory(c);
    }

    public TService Resolve<TService>() where TService : class
    {
        if (!_factories.TryGetValue(
            typeof(TService), out var factory))
        {
            throw new InvalidOperationException(
                $"Сервис {typeof(TService).Name} не зарегистрирован.");
        }

        return (TService)factory(this);
    }
}

// БАЗОВЫЙ КЛАСС CUSTOMER
// Добавлено событие EmailChanged

public class Customer : ICsvExportable, IJsonExportable
{
    private int _customerId;
    private string _name;
    private string _email;
    private string _phone = "не указан";

    public int CustomerId
    {
        get => _customerId;
        set
        {
            if (value <= 0)
                throw new ArgumentException(
                    "ID клиента должен быть больше 0.");

            _customerId = value;
        }
    }

    public string Name
    {
        get => _name;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException(
                    "Имя не может быть пустым.");

            _name = value;
        }
    }

    public string Email
    {
        get => _email;
        set
        {
            if (string.IsNullOrWhiteSpace(value)
                || !value.Contains("@"))
            {
                throw new ArgumentException(
                    "Некорректный формат Email.");
            }

            _email = value;
        }
    }

    public string Phone
    {
        get => _phone;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException(
                    "Телефон не может быть пустым.");

            _phone = value;
        }
    }

    public string City { get; set; } = "не указан";

    public bool IsActive { get; private set; } = true;

    // СОБЫТИЕ: изменение электронной почты
    public event EventHandler<EmailChangedEventArgs> EmailChanged;

    public Customer(int customerId, string name, string email)
    {
        CustomerId = customerId;
        Name = name;
        Email = email;
    }

    public virtual string GetFullName()
    {
        return Name;
    }

    // Перегрузка методов
    public virtual void UpdateEmail(string newEmail)
    {
        UpdateEmail(newEmail, true);
    }

    public virtual void UpdateEmail(
        string newEmail, bool showMessage)
    {
        string oldEmail = Email;

        Email = newEmail;

        if (showMessage)
            Console.WriteLine(
                $"Email успешно обновлен на: {Email}");

        // Генерация события только при реальном изменении
        if (oldEmail != Email)
        {
            EmailChanged?.Invoke(
                this,
                new EmailChangedEventArgs(oldEmail, Email));
        }
    }

    public virtual void ViewProfile()
    {
        string status = IsActive ? "активен" : "неактивен";

        Console.WriteLine(
            $"ID: {CustomerId} | Имя: {GetFullName()} | " +
            $"Email: {Email} | Город: {City} | Статус: {status}");
    }

    public void UpdatePhone(string newPhone)
    {
        Phone = newPhone;
        Console.WriteLine(
            $"Телефон успешно обновлен на: {Phone}");
    }

    public void Deactivate()
    {
        IsActive = false;
        Console.WriteLine(
            $"Клиент '{GetFullName()}' деактивирован.");
    }

    public void Activate()
    {
        IsActive = true;
        Console.WriteLine(
            $"Клиент '{GetFullName()}' активирован.");
    }

    public string GetContactInfo()
    {
        return $"{GetFullName()}: {Email}, " +
               $"тел. {Phone}, город {City}";
    }

    public string GetContactInfo(bool includePhone)
    {
        return includePhone
            ? GetContactInfo()
            : $"{GetFullName()}: {Email}, город {City}";
    }

    // Явная реализация интерфейсов
    string ICsvExportable.Export()
    {
        return $"{CustomerId};{GetFullName()};" +
               $"{Email};{Phone};{City}";
    }

    string IJsonExportable.Export()
    {
        return System.Text.Json.JsonSerializer.Serialize(new
        {
            id = CustomerId,
            name = GetFullName(),
            email = Email,
            city = City
        });
    }
}

// 6. REGULAR CUSTOMER
// Простое наследование

public class RegularCustomer : Customer, INotifiable
{
    public DateTime RegistrationDate { get; set; }
    public DateTime LastEmailUpdate { get; set; }

    public int PurchasesCount { get; private set; }
    public decimal TotalSpent { get; private set; }

    public RegularCustomer(
        int customerId,
        string name,
        string email,
        DateTime registrationDate)
        : base(customerId, name, email)
    {
        RegistrationDate = registrationDate;
    }

    public override void UpdateEmail(
        string newEmail, bool showMessage)
    {
        base.UpdateEmail(newEmail, showMessage);

        LastEmailUpdate = DateTime.Now;

        Console.WriteLine(
            $"[RegularCustomer] Дата обновления Email: " +
            $"{LastEmailUpdate}");
    }

    public override void ViewProfile()
    {
        base.ViewProfile();

        Console.WriteLine(
            $"    Дата регистрации: {RegistrationDate:dd.MM.yyyy} | " +
            $"Покупок: {PurchasesCount} | " +
            $"Потрачено: {TotalSpent:F2} руб.");
    }

    public void MakePurchase(decimal amount)
    {
        if (amount <= 0)
        {
            Console.WriteLine(
                "Сумма покупки должна быть больше 0.");
            return;
        }

        PurchasesCount++;
        TotalSpent += amount;

        Console.WriteLine(
            $"[Покупка] {GetFullName()}: {amount:F2} руб. " +
            $"(всего: {TotalSpent:F2} руб.)");
    }

    public int GetYearsWithUs()
    {
        DateTime today = DateTime.Today;
        int years = today.Year - RegistrationDate.Year;

        if (RegistrationDate.Date > today.AddYears(-years))
            years--;

        return years;
    }

    public string GetStatus()
    {
        return TotalSpent >= 100000m
            ? "Готов к переходу в VIP"
            : "Обычный клиент";
    }

    void INotifiable.Notify(string message)
    {
        Console.WriteLine(
            $"[Email -> {Email}] {message}");
    }
}

// VIP CUSTOMER
// Добавлено событие PointsChanged

public class VipCustomer : Customer, IDiscountable, INotifiable
{
    private int _loyaltyPoints;

    public int LoyaltyPoints
    {
        get => _loyaltyPoints;
        set
        {
            if (value < 0)
                throw new ArgumentException(
                    "Баллы не могут быть отрицательными.");

            _loyaltyPoints = value;
        }
    }

    public DateTime VipSince { get; set; } = DateTime.Now;

    public int TotalEarnedPoints { get; protected set; }

    public virtual decimal DiscountPercent => 10m;

    // СОБЫТИЕ: изменение баллов лояльности
    public event EventHandler<PointsChangedEventArgs> PointsChanged;

    public VipCustomer(
        int customerId,
        string name,
        string email,
        int loyaltyPoints)
        : base(customerId, name, email)
    {
        LoyaltyPoints = loyaltyPoints;
        TotalEarnedPoints = loyaltyPoints;
    }

    // Метод для вызова события
    protected virtual void OnPointsChanged(
        int oldPoints, string reason)
    {
        if (oldPoints != LoyaltyPoints)
        {
            PointsChanged?.Invoke(
                this,
                new PointsChangedEventArgs(
                    oldPoints,
                    LoyaltyPoints,
                    reason));
        }
    }

    public override void ViewProfile()
    {
        Console.WriteLine(
            $"[VIP ПРОФИЛЬ] ID: {CustomerId} | " +
            $"Имя: {GetFullName()} | Email: {Email} | " +
            $"Баллы: {LoyaltyPoints} | " +
            $"Скидка: {DiscountPercent}% | " +
            $"VIP с {VipSince:dd.MM.yyyy}");
    }

    public virtual void AddPoints(int points)
    {
        if (points <= 0)
        {
            Console.WriteLine(
                "Количество баллов должно быть больше 0.");
            return;
        }

        int oldPoints = LoyaltyPoints;

        LoyaltyPoints += points;
        TotalEarnedPoints += points;

        Console.WriteLine(
            $"[VIP] {GetFullName()}: начислено {points} баллов.");

        // Вызов события
        OnPointsChanged(oldPoints, "Начисление баллов");
    }

    public void AddPoints(int points, string reason)
    {
        AddPoints(points);

        if (points > 0)
            Console.WriteLine(
                $"Причина начисления: {reason}");
    }

    public bool SpendPoints(int points)
    {
        if (points <= 0 || points > LoyaltyPoints)
        {
            Console.WriteLine(
                $"[ОШИБКА] Недостаточно баллов у {GetFullName()}.");
            return false;
        }

        int oldPoints = LoyaltyPoints;

        LoyaltyPoints -= points;

        Console.WriteLine(
            $"[VIP] {GetFullName()}: списано {points} баллов.");

        // Вызов события
        OnPointsChanged(oldPoints, "Списание баллов");

        return true;
    }

    public void TransferPoints(
        VipCustomer recipient, int points)
    {
        if (recipient == null || points <= 0)
        {
            Console.WriteLine(
                "Некорректный получатель или количество баллов.");
            return;
        }

        if (ReferenceEquals(this, recipient))
        {
            Console.WriteLine(
                "Нельзя переводить баллы самому себе.");
            return;
        }

        if (LoyaltyPoints < points)
        {
            Console.WriteLine(
                "[ОШИБКА] Недостаточно баллов для перевода.");
            return;
        }

        int oldSenderPoints = LoyaltyPoints;
        int oldRecipientPoints = recipient.LoyaltyPoints;

        LoyaltyPoints -= points;
        recipient.LoyaltyPoints += points;

        Console.WriteLine(
            $"[ПЕРЕВОД] {GetFullName()} перевел {points} " +
            $"баллов клиенту {recipient.GetFullName()}.");

        // Событие у отправителя
        OnPointsChanged(
            oldSenderPoints, "Перевод баллов");

        // Событие у получателя
        recipient.OnPointsChanged(
            oldRecipientPoints, "Получение перевода");
    }

    decimal IDiscountable.ApplyDiscount(decimal price)
    {
        return price - price * DiscountPercent / 100m;
    }

    void INotifiable.Notify(string message)
    {
        NotifyVip(message);
    }

    protected virtual void NotifyVip(string message)
    {
        Console.WriteLine(
            $"[VIP-уведомление для {GetFullName()}] {message}");
    }
}

// PLATINUM CUSTOMER
// Многоуровневое наследование

public class PlatinumCustomer : VipCustomer, IPersonalService
{
    public string PersonalManager { get; set; }

    public int ConciergeRequests { get; private set; }

    public decimal AnnualFee { get; set; }

    public PlatinumCustomer(
        int customerId,
        string name,
        string email,
        int loyaltyPoints,
        string personalManager,
        decimal annualFee)
        : base(customerId, name, email, loyaltyPoints)
    {
        PersonalManager = personalManager;
        AnnualFee = annualFee;
    }

    public override decimal DiscountPercent => 20m;

    public override void AddPoints(int points)
    {
        if (points > 0)
            Console.WriteLine(
                "[PLATINUM] Двойное начисление баллов!");

        base.AddPoints(points > 0 ? checked(points * 2) : points);
    }

    protected override void NotifyVip(string message)
    {
        Console.WriteLine(
            $"[PLATINUM | менеджер: {PersonalManager}] " +
            $"{GetFullName()}: {message}");
    }

    public override void ViewProfile()
    {
        base.ViewProfile();

        Console.WriteLine(
            $"    [PLATINUM] Менеджер: {PersonalManager} | " +
            $"Обращений: {ConciergeRequests} | " +
            $"Годовой взнос: {AnnualFee:F2} руб.");
    }

    void IPersonalService.RequestConcierge(string request)
    {
        ConciergeRequests++;

        Console.WriteLine(
            $"[КОНСЬЕРЖ] {GetFullName()} -> " +
            $"менеджер {PersonalManager}: \"{request}\"");
    }
}

// GROUP CUSTOMER
// КОЛЛЕКЦИЯ List<Customer>

public class GroupCustomer : Customer, IDiscountable
{
    public string GroupName { get; set; }

    public int MaxMembers { get; set; } = 10;

    public DateTime CreatedDate { get; set; } = DateTime.Now;

    // КОЛЛЕКЦИЯ участников
    private readonly List<Customer> _members =
        new List<Customer>();

    public IReadOnlyList<Customer> Members => _members;

    public decimal DiscountPercent =>
        Math.Min(5m + _members.Count, 15m);

    public GroupCustomer(
        int customerId,
        string name,
        string email,
        string groupName)
        : base(customerId, name, email)
    {
        GroupName = groupName;
    }

    public override string GetFullName()
    {
        return GroupName;
    }

    public void AddMember(Customer customer)
    {
        if (customer == null)
            return;

        if (_members.Count >= MaxMembers)
        {
            Console.WriteLine(
                $"[ОШИБКА] Группа '{GroupName}' заполнена.");
            return;
        }

        if (_members.Any(
            member => member.CustomerId == customer.CustomerId))
        {
            Console.WriteLine(
                $"Клиент {customer.GetFullName()} уже в группе.");
            return;
        }

        _members.Add(customer);

        Console.WriteLine(
            $"[ГРУППА] {customer.GetFullName()} " +
            $"добавлен в '{GroupName}'.");
    }

    public bool RemoveMember(Customer customer)
    {
        bool removed = _members.Remove(customer);

        if (removed)
            Console.WriteLine(
                $"[ГРУППА] {customer.GetFullName()} удален.");

        return removed;
    }

    public int GetMembersCount()
    {
        return _members.Count;
    }

    decimal IDiscountable.ApplyDiscount(decimal price)
    {
        return price - price * DiscountPercent / 100m;
    }

    public void NotifyMembers(string message)
    {
        Console.WriteLine(
            $"\n--- Рассылка группы '{GroupName}' ---");

        foreach (Customer member in _members)
        {
            if (member is INotifiable notifiable)
                notifiable.Notify(message);
            else
                Console.WriteLine(
                    $"{member.GetFullName()} не поддерживает уведомления.");
        }
    }
}

// ОБОБЩЕННЫЙ РЕПОЗИТОРИЙ
// КОЛЛЕКЦИИ + ДЕЛЕГАТЫ + СОБЫТИЯ

public class CustomerRepository<T> where T : Customer
{
    // КОЛЛЕКЦИЯ
    private readonly List<T> _items = new List<T>();

    public int Count => _items.Count;

    // СОБЫТИЯ
    public event EventHandler<CustomerEventArgs<T>> CustomerAdded;
    public event EventHandler<CustomerEventArgs<T>> CustomerRemoved;

    public void Add(T customer)
    {
        if (customer == null)
            throw new ArgumentNullException(nameof(customer));

        if (_items.Any(c => c.CustomerId == customer.CustomerId))
        {
            Console.WriteLine(
                $"Клиент с ID {customer.CustomerId} уже существует.");
            return;
        }

        _items.Add(customer);

        // Событие добавления
        CustomerAdded?.Invoke(
            this, new CustomerEventArgs<T>(customer));
    }

    public bool Remove(T customer)
    {
        bool removed = _items.Remove(customer);

        if (removed)
        {
            // Событие удаления
            CustomerRemoved?.Invoke(
                this, new CustomerEventArgs<T>(customer));
        }

        return removed;
    }

    public T FindById(int customerId)
    {
        return _items.Find(
            customer => customer.CustomerId == customerId);
    }

    public List<T> GetAll()
    {
        return new List<T>(_items);
    }

    // ФИЛЬТРАЦИЯ С ПОМОЩЬЮ СОБСТВЕННОГО ДЕЛЕГАТА
    public List<T> Filter(CustomerFilter<T> filter)
    {
        if (filter == null)
            throw new ArgumentNullException(nameof(filter));

        return _items.Where(c => filter(c)).ToList();
    }

    // ГРУППИРОВКА С ПОМОЩЬЮ Dictionary
    public Dictionary<string, List<T>> GroupByCity()
    {
        var groups = new Dictionary<string, List<T>>();

        foreach (T customer in _items)
        {
            string city = customer.City;

            if (!groups.ContainsKey(city))
                groups[city] = new List<T>();

            groups[city].Add(customer);
        }

        return groups;
    }

    // СОРТИРОВКА КОЛЛЕКЦИИ
    public List<T> GetSortedByName()
    {
        return _items
            .OrderBy(c => c.GetFullName())
            .ToList();
    }
}

// СЕРВИС КЛИЕНТОВ (DI)

public class CustomerService
{
    private readonly CustomerRepository<Customer> _repository;
    private readonly ILogger _logger;
    private readonly INotificationSender _sender;

    public CustomerService(
        CustomerRepository<Customer> repository,
        ILogger logger,
        INotificationSender sender)
    {
        _repository = repository
            ?? throw new ArgumentNullException(nameof(repository));

        _logger = logger
            ?? throw new ArgumentNullException(nameof(logger));

        _sender = sender
            ?? throw new ArgumentNullException(nameof(sender));
    }

    public void Register(Customer customer)
    {
        _repository.Add(customer);

        _logger.Log(
            $"Зарегистрирован клиент: {customer.GetFullName()} " +
            $"(ID {customer.CustomerId})");

        _sender.Send(customer.Email, "Добро пожаловать!");
    }

    public void ChangeEmail(int customerId, string newEmail)
    {
        Customer customer = _repository.FindById(customerId);

        if (customer == null)
        {
            _logger.Log($"Клиент с ID {customerId} не найден.");
            return;
        }

        string oldEmail = customer.Email;

        customer.UpdateEmail(newEmail, false);

        _logger.Log(
            $"Email клиента {customer.GetFullName()} изменен: " +
            $"{oldEmail} -> {newEmail}");

        _sender.Send(newEmail, "Ваш email был изменен.");
    }

    public decimal GetPrice(int customerId, decimal price)
    {
        Customer customer = _repository.FindById(customerId);

        if (customer is IDiscountable discountable)
        {
            decimal result = discountable.ApplyDiscount(price);

            _logger.Log(
                $"Скидка {discountable.DiscountPercent}% " +
                $"для {customer.GetFullName()}: " +
                $"{price:F2} -> {result:F2}");

            return result;
        }

        return price;
    }
}

// ВСПОМОГАТЕЛЬНЫЙ КЛАСС

public static class InheritanceHelper
{
    public static string GetChain(Type type)
    {
        List<string> names = new List<string>();

        Type current = type;

        while (current != null && current != typeof(object))
        {
            names.Add(current.Name);
            current = current.BaseType;
        }

        names.Reverse();

        return string.Join(" -> ", names);
    }
}

// ДЕМОНСТРАЦИЯ РАБОТЫ ПРОГРАММЫ

Console.WriteLine("\n=== 1. ПРОСТОЕ НАСЛЕДОВАНИЕ И СОБЫТИЯ ===");

RegularCustomer regular = new RegularCustomer(
    3,
    "Мария Смирнова",
    "maria@mail.com",
    new DateTime(2023, 5, 20))
{
    Phone = "+7 900 111-22-33",
    City = "Москва"
};

// ПОДПИСКА НА СОБЫТИЕ изменения email
regular.EmailChanged += (sender, e) =>
{
    Console.WriteLine($"[СОБЫТИЕ EMAIL] Старый: {e.OldEmail} -> Новый: {e.NewEmail}");
};

Console.WriteLine("Цепочка наследования: " + InheritanceHelper.GetChain(typeof(RegularCustomer)));

regular.ViewProfile();

regular.UpdateEmail("maria.new@mail.com");

regular.UpdatePhone("+7 900 999-88-77");

regular.MakePurchase(35000m);
regular.MakePurchase(80000m);

Console.WriteLine(
    $"Статус: {regular.GetStatus()} | " +
    $"Лет с нами: {regular.GetYearsWithUs()}");

regular.Deactivate();
regular.Activate();

Console.WriteLine(regular.GetContactInfo());

// МНОГОУРОВНЕВОЕ НАСЛЕДОВАНИЕ И СОБЫТИЯ БАЛЛОВ

Console.WriteLine("\n=== 2. VIP, PLATINUM И СОБЫТИЯ ===");

VipCustomer vip = new VipCustomer(1, "Алексей Иванов", "alex@vip.com", 5000)
{
    Phone = "+7 911 000-00-01",
    City = "Санкт-Петербург",
    VipSince = new DateTime(2022, 1, 15)
};

PlatinumCustomer platinum = new PlatinumCustomer(
    2,
    "Елена Петрова",
    "elena@platinum.com",
    12000,
    "Дмитрий Орлов",
    50000m)
{
    Phone = "+7 911 000-00-02",
    City = "Москва",
    VipSince = new DateTime(2021, 3, 10)
};

Console.WriteLine(
    "Цепочка наследования: " +
    InheritanceHelper.GetChain(typeof(PlatinumCustomer)));

// ОБРАБОТЧИК СОБЫТИЯ через делегат EventHandler
EventHandler<PointsChangedEventArgs> pointsHandler =
    (sender, e) =>
    {
        VipCustomer customer = (VipCustomer)sender;

        Console.WriteLine(
            $"[СОБЫТИЕ БАЛЛОВ] {customer.GetFullName()}: " +
            $"{e.OldPoints} -> {e.NewPoints}. " +
            $"Причина: {e.Reason}");
    };

// Подписка на событие
vip.PointsChanged += pointsHandler;
platinum.PointsChanged += pointsHandler;

vip.ViewProfile();
platinum.ViewProfile();

vip.AddPoints(1000);

// У Platinum двойное начисление
platinum.AddPoints(1000);

platinum.SpendPoints(500);

((IPersonalService)platinum).RequestConcierge(
    "Заказать трансфер в аэропорт");

// При переводе вызываются события у двух клиентов
platinum.TransferPoints(vip, 2000);

Console.WriteLine(
    $"Баланс: {vip.GetFullName()} = {vip.LoyaltyPoints}, " +
    $"{platinum.GetFullName()} = {platinum.LoyaltyPoints}");

// МНОЖЕСТВЕННОЕ НАСЛЕДОВАНИЕ ЧЕРЕЗ ИНТЕРФЕЙСЫ

Console.WriteLine("\n=== 3. МНОЖЕСТВЕННОЕ НАСЛЕДОВАНИЕ ===");

Customer plain = new Customer(5, "Иван Сидоров", "ivan@mail.com")
{
    City = "Казань"
};

GroupCustomer group = new GroupCustomer(
    4,
    "Представитель",
    "info@company.com",
    "ООО 'Рога и Копыта'")
{
    City = "Москва",
    MaxMembers = 3
};

// Работа с коллекцией участников
group.AddMember(vip);
group.AddMember(regular);
group.AddMember(platinum);

// Группа заполнена
group.AddMember(plain);

Console.WriteLine($"Участников в группе: {group.GetMembersCount()}");

Console.WriteLine($"Platinum is Customer: {platinum is Customer}");
Console.WriteLine($"Platinum is VipCustomer: {platinum is VipCustomer}");
Console.WriteLine($"Platinum is IDiscountable: {platinum is IDiscountable}");
Console.WriteLine($"Platinum is INotifiable: {platinum is INotifiable}");
Console.WriteLine($"Platinum is IPersonalService: {platinum is IPersonalService}");

// КОЛЛЕКЦИЯ List<Customer> И ПОЛИМОРФИЗМ

Console.WriteLine("\n=== 4. КОЛЛЕКЦИИ И ПОЛИМОРФИЗМ ===");

// КОЛЛЕКЦИЯ разнородных объектов через базовый класс
List<Customer> allCustomers = new List<Customer>
{
    plain,
    regular,
    vip,
    platinum,
    group
};

decimal basePrice = 10000m;

Console.WriteLine("\nСкидки на товар за 10000 руб.:");

foreach (Customer customer in allCustomers)
{
    if (customer is IDiscountable discountable)
    {
        Console.WriteLine(
            $"{customer.GetFullName()}: " +
            $"скидка {discountable.DiscountPercent}% -> " +
            $"{discountable.ApplyDiscount(basePrice):F2} руб.");
    }
    else
    {
        Console.WriteLine(
            $"{customer.GetFullName()}: " +
            $"скидки нет -> {basePrice:F2} руб.");
    }
}

Console.WriteLine("\nПолиморфизм ViewProfile():");

foreach (Customer customer in allCustomers)
{
    customer.ViewProfile();
}

group.NotifyMembers("Скидка 20% на новые услуги для всех участников группы!");

// ПЕРЕГРУЗКА МЕТОДОВ

Console.WriteLine("\n=== 5. ПЕРЕГРУЗКА МЕТОДОВ ===");

plain.UpdateEmail("ivan.new@mail.com", false);

Console.WriteLine(plain.GetContactInfo(false));

vip.AddPoints(50, "Бонус за участие в акции");

// ОБОБЩЕННЫЙ РЕПОЗИТОРИЙ И СОБЫТИЯ КОЛЛЕКЦИИ

Console.WriteLine("\n=== 6. РЕПОЗИТОРИЙ И СОБЫТИЯ ===");

CustomerRepository<Customer> customerRepository =
    new CustomerRepository<Customer>();

// Обработчик добавления клиента
EventHandler<CustomerEventArgs<Customer>> addedHandler =
    (sender, e) =>
    {
        Console.WriteLine(
            $"[СОБЫТИЕ] Клиент добавлен: " +
            $"{e.Customer.GetFullName()} " +
            $"(ID: {e.Customer.CustomerId})");
    };

// Обработчик удаления клиента
EventHandler<CustomerEventArgs<Customer>> removedHandler =
    (sender, e) =>
    {
        Console.WriteLine($"[СОБЫТИЕ] Клиент удален: {e.Customer.GetFullName()}");
    };

customerRepository.CustomerAdded += addedHandler;
customerRepository.CustomerRemoved += removedHandler;

customerRepository.Add(plain);
customerRepository.Add(regular);
customerRepository.Add(vip);
customerRepository.Add(platinum);
customerRepository.Add(group);

Console.WriteLine($"Всего клиентов: {customerRepository.Count}");

Customer foundCustomer = customerRepository.FindById(2);

Console.WriteLine(foundCustomer != null ? $"Найден клиент: {foundCustomer.GetFullName()}" : "Клиент не найден");

// Демонстрация удаления и соответствующего события
Customer temporary = new Customer(6, "Временный клиент", "temp@mail.com");

customerRepository.Add(temporary);
customerRepository.Remove(temporary);

// Отписка от событий
customerRepository.CustomerAdded -= addedHandler;
customerRepository.CustomerRemoved -= removedHandler;

// Отдельный репозиторий для VIP
CustomerRepository<VipCustomer> vipRepository = new CustomerRepository<VipCustomer>();

vipRepository.Add(vip);
vipRepository.Add(platinum);

Console.WriteLine($"VIP-клиентов в репозитории: {vipRepository.Count}");


// ДЕЛЕГАТЫ И ФИЛЬТРАЦИЯ КОЛЛЕКЦИЙ

Console.WriteLine("\n=== 7. ДЕЛЕГАТЫ И ФИЛЬТРАЦИЯ ===");

// Собственный делегат CustomerFilter<T>
CustomerFilter<Customer> vipFilter =customer => customer is VipCustomer;

List<Customer> filteredVip =customerRepository.Filter(vipFilter);

Console.WriteLine("\nКлиенты VIP и Platinum:");

foreach (Customer customer in filteredVip)
{
    Console.WriteLine($"- {customer.GetFullName()}");
}

// Еще один делегат для фильтрации
CustomerFilter<Customer> moscowFilter =customer => customer.City == "Москва";

List<Customer> moscowCustomers =customerRepository.Filter(moscowFilter);

Console.WriteLine("\nКлиенты из Москвы:");

foreach (Customer customer in moscowCustomers)
{
    Console.WriteLine($"- {customer.GetFullName()}");
}

// ВСТРОЕННЫЙ ДЕЛЕГАТ Action<T>
Console.WriteLine("\nДемонстрация Action<Customer>:");

Action<Customer> printName = customer => Console.WriteLine($"Имя: {customer.GetFullName()}");

Action<Customer> printEmail = customer => Console.WriteLine($"Email: {customer.Email}");

// Многоадресный делегат
Action<Customer> combinedAction = printName + printEmail;

combinedAction(vip);

// Удаление одного метода из делегата
combinedAction -= printEmail;

Console.WriteLine("После удаления второго действия:");

combinedAction(regular);

// DICTIONARY И СОРТИРОВКА

Console.WriteLine("\n=== 8. DICTIONARY И СОРТИРОВКА ===");

// Коллекция Dictionary: город -> список клиентов
Dictionary<string, List<Customer>> customersByCity =
    customerRepository.GroupByCity();

foreach (var cityGroup in customersByCity)
{
    Console.WriteLine($"\nГород: {cityGroup.Key}");

    foreach (Customer customer in cityGroup.Value)
    {
        Console.WriteLine($"  - {customer.GetFullName()}");
    }
}

// Сортировка с помощью LINQ
Console.WriteLine("\nКлиенты по алфавиту:");

List<Customer> sortedCustomers =customerRepository.GetSortedByName();

foreach (Customer customer in sortedCustomers)
{
    Console.WriteLine($"- {customer.GetFullName()}");
}
// ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСОВ

Console.WriteLine("\n=== 9. ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСОВ ===");

IDiscountable vipDiscount = vip;

Console.WriteLine($"VIP: цена 1000 -> {vipDiscount.ApplyDiscount(1000m):F2} руб.");

Console.WriteLine("CSV: " + ((ICsvExportable)vip).Export());

Console.WriteLine("JSON: " + ((IJsonExportable)vip).Export());

Console.WriteLine("CSV группы: " + ((ICsvExportable)group).Export());

((INotifiable)regular).Notify("Ваш заказ отправлен.");

((INotifiable)vip).Notify("Вам начислены бонусные баллы.");

((INotifiable)platinum).Notify("Ваш менеджер свяжется с вами.");

// УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ

Console.WriteLine("\n=== 10. УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ (DI) ===");

var container = new ServiceContainer();

container.RegisterSingleton<ILogger>(c => new ConsoleLogger());

container.RegisterSingleton<INotificationSender>(c => new EmailSender());

container.RegisterSingleton<CustomerRepository<Customer>>(c => new CustomerRepository<Customer>());

container.RegisterTransient<CustomerService>(
    c => new CustomerService(
        c.Resolve<CustomerRepository<Customer>>(),
        c.Resolve<ILogger>(),
        c.Resolve<INotificationSender>()));

CustomerService customerService =
    container.Resolve<CustomerService>();

// Событие изменения Email будет работать и через сервис
plain.EmailChanged += (sender, e) =>
{
    Console.WriteLine($"[СОБЫТИЕ ИЗ СЕРВИСА] {e.OldEmail} -> {e.NewEmail}");
};

customerService.Register(plain);
customerService.Register(vip);
customerService.Register(platinum);

customerService.ChangeEmail(5, "ivan.service@mail.com");

customerService.GetPrice(1, 10000m);
customerService.GetPrice(2, 10000m);
customerService.GetPrice(5, 10000m);

// Подмена зависимости: Email -> SMS
Console.WriteLine("\n--- Замена EmailSender на SmsSender ---");

container.RegisterSingleton<INotificationSender>(c => new SmsSender());

CustomerService smsService =container.Resolve<CustomerService>();

smsService.ChangeEmail(1, "alex.new@vip.com");

// ЗАКЛЮЧИТЕЛЬНАЯ ДЕМОНСТРАЦИЯ

Console.WriteLine("\n=== 11. ИТОГОВАЯ СТАТИСТИКА ===");

Console.WriteLine($"Всего клиентов: {customerRepository.Count}");

Console.WriteLine($"VIP и Platinum: {filteredVip.Count}");

Console.WriteLine($"Клиентов из Москвы: {moscowCustomers.Count}");

Console.WriteLine($"Количество городов: {customersByCity.Count}");

Console.WriteLine($"Баллы VIP-клиента: {vip.LoyaltyPoints}");

Console.WriteLine($"Баллы Platinum-клиента: {platinum.LoyaltyPoints}");

// Отписка от обработчика событий баллов
vip.PointsChanged -= pointsHandler;
platinum.PointsChanged -= pointsHandler;



=== 1. ПРОСТОЕ НАСЛЕДОВАНИЕ И СОБЫТИЯ ===
Цепочка наследования: Customer -> RegularCustomer
ID: 3 | Имя: Мария Смирнова | Email: maria@mail.com | Город: Москва | Статус: активен
    Дата регистрации: 20.05.2023 | Покупок: 0 | Потрачено: 0,00 руб.
Email успешно обновлен на: maria.new@mail.com
[СОБЫТИЕ EMAIL] Старый: maria@mail.com -> Новый: maria.new@mail.com
[RegularCustomer] Дата обновления Email: 18.11.2026 22:07:19
Телефон успешно обновлен на: +7 900 999-88-77
[Покупка] Мария Смирнова: 35000,00 руб. (всего: 35000,00 руб.)
[Покупка] Мария Смирнова: 80000,00 руб. (всего: 115000,00 руб.)
Статус: Готов к переходу в VIP | Лет с нами: 3
Клиент 'Мария Смирнова' деактивирован.
Клиент 'Мария Смирнова' активирован.
Мария Смирнова: maria.new@mail.com, тел. +7 900 999-88-77, город Москва

=== 2. VIP, PLATINUM И СОБЫТИЯ ===
Цепочка наследования: Customer -> VipCustomer -> PlatinumCustomer
[VIP ПРОФИЛЬ] ID: 1 | Имя: Алексей Иванов | Email: alex@vip.com | Баллы: 5000 | Скидка: 10% | VIP с 15.01.2